In [2]:
#import necessary libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [3]:
#load the data
dataset_path = 'PetImages'


In [4]:
#image preprocessing
datagen = ImageDataGenerator(
    #normalize pixels
    rescale = 1./255,
    #data augmentation
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    #split dataset
    validation_split = 0.2 #calidation split to create train and validation data
)

In [5]:
#load training data
#training data
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size = (128, 128),
    batch_size = 32,
    class_mode = 'binary',
    subset = 'training'
)

Found 20000 images belonging to 2 classes.


In [6]:
#load validation data
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size = (128, 128),
    batch_size = 32,
    class_mode = 'binary',
    subset = 'validation'
)

Found 4998 images belonging to 2 classes.


In [7]:
print(train_data.class_indices)

{'Cat': 0, 'Dog': 1}


In [8]:
#build the model
model = tf.keras.Sequential()

In [9]:
#first convolutional layer
model.add(
    tf.keras.layers.Conv2D(
        filters = 32,
        kernel_size = (3, 3),
        activation = 'relu',
        input_shape = (128, 128, 3)
    )
)

c:\Users\bsais\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
#first pooling layer
model.add(
    tf.keras.layers.MaxPooling2D(
        pool_size = (2, 2)
    )
)

In [11]:
#second convolutional layer
model.add(
    tf.keras.layers.Conv2D(
        filters = 32,
        kernel_size = (3, 3),
        activation = 'relu'
    )
)

In [12]:
#second pooling layer
model.add(
    tf.keras.layers.MaxPooling2D(
        pool_size = (2, 2)
    )
)

In [13]:
#flatten layer
model.add(
    tf.keras.layers.Flatten()
)

In [14]:
#dense layer
model.add(
    tf.keras.layers.Dense(
        units = 128,
        activation = 'relu'
    )
)

In [15]:
#output layer
model.add(
    tf.keras.layers.Dense(
        units = 1,
        activation = 'sigmoid'
    )
)

In [16]:
#view model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,696,801 (14.10 MB)

 Trainable params: 3,696,801 (14.10 MB)

 Non-trainable params: 0 (0.00 B)

#### (none : batch size, 126 : height, 126 : width, channels : no. of filters (32))

input = 128 * 128 * 3

filter = 3 * 3

no. of filters = 32

output size formula : 
* output = ((input - kernel) / stride) + 1
* stride decides the no. of steps
* output = ((123 - 3) / 1) + 1 => 126

height : 126

weidth : 126

channels : 32

final output : (None, 126, 126, 32)

#### kernel

formula : (kernel_height *kernel_width * InputChannels + 1) * filters

kernel = 3 * 3

inpurclannels = 3

filters : 32

params = (3 * 3 * 3 + 1) * 32
       = (27 + 1) * 32
       = 28 * 32
       = 896

#### pooling2d

MaxPooling2D((2,2))

pooling = no learning, no weights (only reduces size)

input = 126 * 126 * 32

pool = 2 * 2

formula : 126/2 = 63

output : 63 * 63 * 32

final : (None, 63, 63, 32)

#### second convo2d

input from pooling : (none, 63 * 63 * 32)

kernel : 3 * 3

output : 63 - 3 + 1 = 61

(none, 61, 61, 32)

param = 9248

kernels = (3 * 3 * 32 + 1) * 32
        = (288 + 1) * 32
        = 289 * 32
        = 9248

#### second pooling2d

(none, 30, 30, 32)

param = 0

input = (61, 61, 32)

pqram = 0

pooling = 2 * 2

floor division : 61 / 2 = 30

final : (none, 30, 30, 32)
v1 -> pool1
no params

#### flatten

input : 30 * 30 * 32

output : 30 * 30 * 32 = 28000 (numbers)

flatten : it will convert 3d -> 1d vector (flatten only reshapes)

#### dense layer

input + 28000

neurons = 128

parameters = (input * neurons) + bias
           = (28800 * 128) + 128
           = 3686400 + 128
           = 3686528

#### output layer

input = 128

output = 1

output shape (none, 1)

meaning = single prediction

--------> cat or dog

parameters calculation : 
        = 128 * 1 + 1
        = 129

#### input -> conv1 -> pool1 -> conv2 -> pool -> flatten -> dense -> output

In [17]:
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [18]:
#compile the model
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [19]:
#train the model
model.fit(
    train_data,
    epochs = 10,
    validation_data = val_data
)

Epoch 1/10
 82/625 ━━━━━━━━━━━━━━━━━━━━ 2:32 281ms/step - accuracy: 0.5249 - loss: 0.8783

c:\Users\bsais\anaconda3\Lib\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Truncated File Read
  warnings.warn(str(msg))


625/625 ━━━━━━━━━━━━━━━━━━━━ 145s 228ms/step - accuracy: 0.6374 - loss: 0.6449 - val_accuracy: 0.7041 - val_loss: 0.5766
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 124s 198ms/step - accuracy: 0.7114 - loss: 0.5574 - val_accuracy: 0.7139 - val_loss: 0.5519
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 122s 196ms/step - accuracy: 0.7558 - loss: 0.4994 - val_accuracy: 0.7691 - val_loss: 0.4782
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 123s 196ms/step - accuracy: 0.7817 - loss: 0.4595 - val_accuracy: 0.7865 - val_loss: 0.4633
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 122s 195ms/step - accuracy: 0.7970 - loss: 0.4311 - val_accuracy: 0.7933 - val_loss: 0.4405
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 123s 197ms/step - accuracy: 0.8112 - loss: 0.4071 - val_accuracy: 0.7931 - val_loss: 0.4389
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 123s 196ms/step - accuracy: 0.8231 - loss: 0.3873 - val_accuracy: 0.8019 - val_loss: 0.4314
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 124s 199ms/step - accuracy: 0.8310 - loss: 0.37

In [20]:
#evaluate the model
loss, accuracy = model.evaluate(val_data)
print("accuracy: ", accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 41s 261ms/step - accuracy: 0.8231 - loss: 0.3896
accuracy:  0.8231292366981506


In [21]:
model.save('cat_dog_classifier.keras')